In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\03-agents\\02-agent-basics\\')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\03-agents\02-agent-basics


In [2]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from markitdown import MarkItDown
from typing import Any, Dict, Iterable, List
import json
from pydantic import BaseModel, Field
from typing import Literal
from minsearch import AppendableIndex
from gitsource import GithubRepositoryDataReader, chunk_documents

In [3]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [4]:
from openai import OpenAI
openai_client  = OpenAI()

In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [6]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [7]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [8]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [9]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [10]:
question = "How do I create a dahsbord in Evidently?"

In [11]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1




# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )
    
        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)
    
        has_function_calls = False
    
        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True
    
            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)
    
        iteration_number = iteration_number + 1
        print()
        
        if not has_function_calls:
            break

iteraration number 1...
ASSISTANT: To provide an accurate and comprehensive response regarding dashboard creation, I will first search for general documentation about dashboard creation processes. This initial step will help identify foundational information and detailed guidelines on how to create and customize a dashboard.

Let's proceed with the search.
executing search({"query":"create a dashboard"})...

iteraration number 2...
ASSISTANT: I found relevant information about creating dashboards, including adding tabs and panels. Here’s a summary of the key points:

1. **Dashboards** allow you to create panels for visualizing evaluation results over time, but you must first add reports with evaluation results to your project.
2. **Adding Tabs**: You can add multiple tabs to organize panels by entering "Edit" mode and clicking on the plus sign to add a new tab. You can also create pre-built templates if related metrics are available.
3. **Adding Panels**: You can add various types of p

In [12]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name
    
        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'
        
        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]
            
        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0
    
        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )
        
            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)
        
            has_function_calls = False
        
            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True
        
                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)
        
            iteration_number = iteration_number + 1
            print()
            
            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]
        
        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break
            
            message_history = self.loop(user_prompt, message_history)

In [13]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [14]:

messages = agent.loop('evidently adshboards') 

iteraration number 0...
ASSISTANT: I’ll start by performing a search for "dashboards" to see if there are any specific entries related to that topic. Dashboards are often critical in data visualization and monitoring, so it's essential to find the relevant documentation to understand their functionalities and features.

Let's do that now.
executing search({"query":"dashboards"})...

iteraration number 1...
ASSISTANT: The search yielded several entries related to dashboards, emphasizing how to manage panels, add tabs, and the overall functionality of dashboards. Here are some key points extracted from the results:

1. **Overview of Dashboards**: Dashboards allow users to visualize evaluation results over time. Each project has its own dashboard, which must first be populated with reports to display valuable insights.

2. **Adding and Managing Panels**:
   - You can add panels through both the UI and the Python API.
   - Various types of panels can be created, including text panels, coun